# Task 4 — Graph Topology Ingestion into Neo4j

## Approach and reasoning

The lab requires the graph topology to reach Neo4j **directly from Kafka, with no
intermediate Spark layer**. We therefore use the Neo4j Connector for Kafka in
sink mode, with the **Cypher strategy**: each topic is bound to a Cypher
statement that the connector executes per batch of messages.

### How idempotency is achieved
Every statement uses `MERGE` keyed on our stable id, never `CREATE`:

```cypher
// cpg.nodes
MERGE (n:CpgNode {id: __value.node.id})
SET n.type = __value.node.type,
    n.file_id = __value.file_id,
    n.rel_path = __value.rel_path,
    n.file_hash = __value.file_hash

// cpg.edges — note both endpoints are MERGEd
MERGE (s:CpgNode {id: __value.edge.src_id})
MERGE (d:CpgNode {id: __value.edge.dst_id})
MERGE (s)-[r:CPG_EDGE {id: __value.edge.id}]->(d)
SET r.type = __value.edge.type, r.file_hash = __value.file_hash
```

**Why both endpoints are MERGEd.** Node and edge events travel on separate
topics with independent partitions, so an edge can arrive before its endpoints.
`MATCH` would silently drop those edges; `MERGE` creates a placeholder that the
node event later enriches. This removed an ordering dependency between two
topics that we could not otherwise guarantee.

A uniqueness constraint on `CpgNode.id` makes each `MERGE` an index lookup rather
than a scan, and makes duplicate nodes impossible at the database level.

### Step 1 — create the constraint (run once, before the connectors)

In [ ]:
import os
os.chdir("..")
!docker exec -i neo4j cypher-shell -u neo4j -p password < src/neo4j/constraints.cypher

### Step 2 — confirm the Neo4j connector plugin is installed

In [ ]:
!curl -s http://localhost:8083/connector-plugins | python -m json.tool | grep -i neo4j

### Step 3 — register both sink connectors

In [ ]:
!curl -s -X POST http://localhost:8083/connectors \
  -H 'Content-Type:application/json' -d @src/neo4j/neo4j-sink-nodes.json | python -m json.tool
!curl -s -X POST http://localhost:8083/connectors \
  -H 'Content-Type:application/json' -d @src/neo4j/neo4j-sink-edges.json | python -m json.tool

### Step 4 — verify both connectors and their tasks are RUNNING

In [ ]:
!curl -s http://localhost:8083/connectors/neo4j-sink-cpg-nodes/status | python -m json.tool
!curl -s http://localhost:8083/connectors/neo4j-sink-cpg-edges/status | python -m json.tool

> A connector can report `RUNNING` while an individual task has failed. Check the
> `tasks` array in the output above, not just the top-level state.

### Step 5 — run the parser against the live broker

In [ ]:
!python src/parser/parser_service.py --manifest reports/file_manifest.json \
    --repo ./optimum --bootstrap localhost:9092

### Step 6 — verify the graph in Neo4j

In [ ]:
!docker exec neo4j cypher-shell -u neo4j -p password \
  "MATCH (n:CpgNode) RETURN count(n) AS nodes"
!docker exec neo4j cypher-shell -u neo4j -p password \
  "MATCH ()-[r:CPG_EDGE]->() RETURN r.type AS type, count(*) AS n ORDER BY n DESC"

### Step 7 — the duplicate check that must return zero rows

In [ ]:
!docker exec neo4j cypher-shell -u neo4j -p password \
  "MATCH (n:CpgNode) WITH n.id AS id, count(*) AS c WHERE c > 1 RETURN id, c"

### Database UI evidence

Open the Neo4j Browser at <http://localhost:7474> and run:

```cypher
MATCH (n:CpgNode)-[r:CPG_EDGE]->(m) RETURN n, r, m LIMIT 100
```

**Insert your screenshots here** (save them under `jupyter-book/images/`):

![Neo4j graph view](images/neo4j_graph.png)

![Neo4j edge breakdown](images/neo4j_edge_counts.png)

## Reflection

*(Replace this with what actually happened on your machine.)*

**What failed.** Our first edge statement used `MATCH` for the endpoints. Roughly
a third of edges vanished, because edge events reached the sink before the
corresponding node events from the other topic. Switching to `MERGE` on both
endpoints fixed it and made the sink order-independent.

**What worked.** Creating the uniqueness constraint *before* starting the
connectors. Without it, `MERGE` degrades to a full label scan and ingestion of
tens of thousands of nodes becomes unusably slow.